# Safe Query Execution

**Notebook Version**: 1.0.0  
**Compatible with DBMCP**: 0.1.0+  
**Last Updated**: 2026-01-26  
**Test Database Version**: 1.0  
**Estimated Time**: 15 minutes  
**Difficulty**: Intermediate

## Overview

This notebook demonstrates how to safely execute ad-hoc SQL queries using the DBMCP query execution tool. You'll learn about read-only enforcement, row limits, result formatting, and error handling.

## Prerequisites

- Completed [01_basic_connection.ipynb](01_basic_connection.ipynb) or familiar with database connections
- Basic understanding of SQL SELECT queries
- Familiarity with database concepts: tables, columns, joins

## What You'll Learn

- How to execute SELECT queries safely with automatic row limiting
- How read-only enforcement protects against accidental writes
- How to handle query results and interpret response structure
- How to work with aggregation and join queries
- How to handle errors and blocked operations gracefully

## Environment Verification

In [ ]:
import sys
from pathlib import Path

print(f"Python version: {sys.version}")

# Find repository root and add to path
def find_repo_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    if current.name == "notebooks":
        return current.parent.parent
    return current

repo_root = find_repo_root()
sys.path.insert(0, str(repo_root))
print(f"Repository root: {repo_root}")

from examples.shared.notebook_helpers import verify_notebook_environment

verify_notebook_environment()

## Section 1: Connection Setup

Quick connection setup (refer to notebook 01 for details).

In [ ]:
from sqlalchemy import create_engine

from examples.shared.notebook_helpers import print_table
from src.db.query import QueryService

# Connect to test database
test_db_path = repo_root / "examples" / "test_database" / "example.db"
engine = create_engine(f"sqlite:///{test_db_path}")

# Create a connection_id (normally this comes from connect_database)
connection_id = "test_connection_123"

# Create query service (takes engine, not connection)
query_svc = QueryService(engine)

print(f"Connection ID: {connection_id}")
print(f"Connected to: {test_db_path.name}")

## Section 2: Basic SELECT Query

Let's start with a simple SELECT query to retrieve customer data.

**What we'll do**:
1. Execute a basic SELECT query
2. Examine the result structure
3. Understand row limiting

In [ ]:
# Execute a simple SELECT query
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="SELECT customer_id, first_name, last_name, email FROM customers",
    row_limit=5  # Limit to 5 rows for display
)

# Get formatted results
results = query_svc.get_query_results(query)

print(f"Query Status: {results['status']}")
print(f"Query Type: {results['query_type']}")
print(f"Execution Time: {results['execution_time_ms']}ms")
print(f"Rows Returned: {results['rows_returned']}")
print(f"Limited: {results.get('limited', False)}")
print()

# Display results as table
if results['status'] == 'success' and results['rows']:
    columns = results['columns']
    rows = [[row[col] for col in columns] for row in results['rows']]
    print_table(rows, columns)

**Understanding the results**:

The query result includes several important fields:
- **status**: 'success', 'blocked', or 'error'
- **query_type**: Detected type (select, insert, update, delete, other)
- **execution_time_ms**: How long the query took
- **rows_returned**: Number of rows in the response
- **limited**: Whether results were truncated due to row_limit

**Key takeaways**:
- Row limits protect against accidentally retrieving millions of rows
- Execution time helps identify slow queries
- The 'limited' flag tells you if there's more data available

## Section 3: Query with WHERE Clause

Let's execute a filtered query using a WHERE clause.

**What we'll do**:
1. Execute a query with filtering conditions
2. See how specific data can be retrieved

In [ ]:
# Query with WHERE clause
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="SELECT * FROM customers WHERE customer_id <= 3",
    row_limit=10
)

results = query_svc.get_query_results(query)

print(f"Status: {results['status']}")
print(f"Rows returned: {results['rows_returned']}")
print()

if results['rows']:
    for i, row in enumerate(results['rows'], 1):
        print(f"Customer {i}: {row.get('first_name', 'N/A')} {row.get('last_name', 'N/A')} ({row.get('email', 'N/A')})")

## Section 4: Aggregation Queries

SQL aggregation functions (COUNT, SUM, AVG, etc.) are fully supported.

**What we'll do**:
1. Execute COUNT queries
2. Use GROUP BY for summaries
3. Interpret aggregation results

In [ ]:
# Count total customers
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="SELECT COUNT(*) as total_customers FROM customers",
    row_limit=100
)

results = query_svc.get_query_results(query)

if results['status'] == 'success':
    total = results['rows'][0]['total_customers']
    print(f"Total customers in database: {total}")

In [ ]:
# Orders per customer using GROUP BY
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="""
        SELECT customer_id, COUNT(*) as order_count
        FROM orders
        GROUP BY customer_id
        ORDER BY order_count DESC
    """,
    row_limit=10
)

results = query_svc.get_query_results(query)

print("Orders per Customer:")
print("=" * 30)
if results['rows']:
    for row in results['rows']:
        print(f"  Customer {row['customer_id']}: {row['order_count']} orders")

## Section 5: JOIN Queries

JOINs combine data from multiple tables. Let's see how they work with the query executor.

**What we'll do**:
1. Execute an INNER JOIN query
2. Combine customer and order data
3. Handle multiple table results

In [ ]:
# Join customers with their orders
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="""
        SELECT 
            c.first_name,
            c.last_name,
            o.order_id,
            o.order_date,
            o.total_amount
        FROM customers c
        INNER JOIN orders o ON c.customer_id = o.customer_id
        ORDER BY o.order_date DESC
    """,
    row_limit=10
)

results = query_svc.get_query_results(query)

print("Recent Orders with Customer Names:")
print("=" * 60)
if results['rows']:
    columns = results['columns']
    rows = [[row[col] for col in columns] for row in results['rows']]
    print_table(rows, columns)

## Section 6: Read-Only Enforcement

The query executor blocks write operations (INSERT, UPDATE, DELETE) by default. This is a safety feature to prevent accidental data modification.

**What we'll do**:
1. Attempt a DELETE query and see it blocked
2. Attempt an UPDATE query and see it blocked
3. Understand the error messages

In [ ]:
# Attempt a DELETE query (will be blocked)
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="DELETE FROM customers WHERE customer_id = 9999",
    row_limit=100
)

results = query_svc.get_query_results(query)

print("DELETE Query Result:")
print(f"  Status: {results['status']}")
print(f"  Query Type: {results['query_type']}")
print(f"  Allowed: {results['is_allowed']}")
if 'error_message' in results:
    print(f"  Message: {results['error_message']}")

In [ ]:
# Attempt an UPDATE query (will be blocked)
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="UPDATE customers SET email='test@test.com' WHERE customer_id = 1",
    row_limit=100
)

results = query_svc.get_query_results(query)

print("UPDATE Query Result:")
print(f"  Status: {results['status']}")
print(f"  Query Type: {results['query_type']}")
print(f"  Allowed: {results['is_allowed']}")
if 'error_message' in results:
    print(f"  Message: {results['error_message']}")

**Understanding the results**:

Write operations are blocked with clear messages:
- **status: 'blocked'** indicates the query was prevented
- **is_allowed: False** confirms write protection is active
- **error_message** explains why and how to enable writes if needed

**Key takeaways**:
- Read-only mode protects production data from accidents
- Query type detection works on INSERT, UPDATE, DELETE, and DDL
- Clear error messages help users understand what happened

## Section 7: Row Limit Configuration

The row_limit parameter controls how many rows are returned. Let's see how different limits affect results.

**What we'll do**:
1. Compare results with different row limits
2. See how the 'limited' flag indicates truncation
3. Understand the default and maximum limits

In [ ]:
# Test different row limits
for limit in [3, 5, 10]:
    query = query_svc.execute_query(
        connection_id=connection_id,
        query_text="SELECT * FROM customers",
        row_limit=limit
    )
    results = query_svc.get_query_results(query)

    print(f"Limit={limit}: Returned {results['rows_returned']} rows, Limited={results.get('limited', False)}")

In [ ]:
# Test invalid row limits
print("Testing invalid row limits:")
print()

# Too small
try:
    query = query_svc.execute_query(
        connection_id=connection_id,
        query_text="SELECT * FROM customers",
        row_limit=0
    )
except ValueError as e:
    print(f"  row_limit=0: {e}")

# Too large
try:
    query = query_svc.execute_query(
        connection_id=connection_id,
        query_text="SELECT * FROM customers",
        row_limit=10001
    )
except ValueError as e:
    print(f"  row_limit=10001: {e}")

**Row Limit Guidelines**:

| Scenario | Recommended Limit |
|----------|------------------|
| Quick inspection | 5-10 rows |
| Data exploration | 100-500 rows |
| Export/analysis | 1000-5000 rows |
| Maximum allowed | 10,000 rows |

**Default**: 1,000 rows if not specified

## Section 8: Error Handling

Let's see how errors are handled for invalid queries.

**What we'll do**:
1. Execute a query with syntax error
2. Query a non-existent table
3. Interpret error messages

In [ ]:
# Query non-existent table
query = query_svc.execute_query(
    connection_id=connection_id,
    query_text="SELECT * FROM nonexistent_table",
    row_limit=10
)

results = query_svc.get_query_results(query)

print("Query to non-existent table:")
print(f"  Status: {results['status']}")
if 'error_message' in results:
    print(f"  Error: {results['error_message'][:100]}...")

In [ ]:
# Empty query
try:
    query = query_svc.execute_query(
        connection_id=connection_id,
        query_text="",
        row_limit=10
    )
except ValueError as e:
    print(f"Empty query error: {e}")

**Error handling best practices**:

1. Always check `results['status']` before accessing data
2. Handle both 'blocked' and 'error' statuses
3. Use `error_message` for debugging
4. Catch ValueError for parameter validation errors

## Summary

**What we covered**:
- Executed SELECT queries safely with automatic row limiting
- Used WHERE clauses, aggregations, and JOINs
- Demonstrated read-only enforcement blocking write operations
- Configured row limits and understood their boundaries
- Handled errors and invalid queries gracefully

**Key concepts**:
- **Read-only by default**: Write operations are blocked for safety
- **Row limiting**: Automatic protection against runaway queries
- **Structured results**: Consistent response format for all queries
- **Error handling**: Clear status and error messages

**Common pitfalls**:
- Not checking query status before using results
- Setting row limits too high for large tables
- Forgetting that write operations are blocked by default

## Next Steps

**Continue learning**:
- [01_basic_connection.ipynb](01_basic_connection.ipynb) - Review connection basics
- [02_table_inspection.ipynb](02_table_inspection.ipynb) - Explore table schemas
- [03_advanced_patterns.ipynb](03_advanced_patterns.ipynb) - Relationship inference
- [04_column_analysis.ipynb](04_column_analysis.ipynb) - Column purpose analysis
- [05_documentation_cache.ipynb](05_documentation_cache.ipynb) - Documentation caching

**Explore further**:
- Try complex queries with subqueries and CTEs
- Experiment with different row limit values
- Use query results for data analysis

**Need help?**
- [Report issues](https://github.com/anthropics/dbmcp/issues)
- [Join discussions](https://github.com/anthropics/dbmcp/discussions)